In [1]:
import polars as pl
import sys
from pathlib import Path
import os

In [ ]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [ ]:
DAY = 86400
WEEK = 7 * DAY

max_time = transactions.select(pl.max("time")).item()

valid_start = max_time - WEEK
train_start = valid_start - 12 * WEEK   # 12 周

train = transactions.filter(
    (pl.col("time") >= train_start) &
    (pl.col("time") < valid_start)
)

valid= transactions.filter(
    pl.col("time") >= valid_start
)


In [ ]:
valid_pos = (
    valid
    .select(["customer_id", "article_id"])
    .unique()
    .with_columns(pl.lit(1).alias("label"))
)

In [ ]:
recall=pl.read_parquet('/home/mingyu/Recommand-System/项目/kaggle-H&M/save/ranker/predict_recall_result.parquet')

In [ ]:
train_labeled = (
    recall
    .join(
        valid_pos,
        on=["customer_id", "article_id"],
        how="left"
    )
    .with_columns(
        pl.col("label").fill_null(0).cast(pl.UInt8)
    )
)

In [ ]:
train_labeled.write_parquet('data_labeled.parquet')